# CHM performance by GEDI RH95 height class

This notebook compares the locally trained model with existing canopy-height products **on exactly the same GEDI shots within each study area**. The analysis is descriptive at product level; it is not a controlled comparison of model architectures.

The first height class is labelled 0--5 m for consistency with the manuscript, although the retained evaluation domain starts at 2 m. Small sample sizes in tall classes are reported explicitly. They can increase uncertainty, but they are not assumed to be the sole cause of tall-canopy errors.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
SOURCE = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "CHM_Comparison" / "GEDI_TEST_product_valid_support_signed_errors.csv.gz"
OUT = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "CHM_Height_Class_Comparison_Strict_Common"
TABLES, FIGURES = OUT / "tables", OUT / "figures"
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)
assert SOURCE.is_file(), SOURCE

PRODUCTS = {
    "Ifran": ["Our B4 Phase 2", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"],
    "Maamoura": ["Our B4 Phase 2", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"],
    "Agadir": ["Our B4 Phase 2", "Pauls 2020", "Lang 2020", "Meta/Tolan 2023"],
}
EXTENDED_PRODUCTS = {
    "Ifran": PRODUCTS["Ifran"] + ["GFCH 2019"],
    "Maamoura": PRODUCTS["Maamoura"] + ["GFCH 2019"],
    "Agadir": PRODUCTS["Agadir"],  # P21 is not evaluable at Agadir.
}
LABEL = {"Our B4 Phase 2":"Our model", "Pauls 2020":"Pa24 (Pauls et al.)", "Lang 2020":"L23 (Lang et al.)", "Meta/Tolan 2023":"T24 (Tolan et al.)", "GFCH 2019":"P21 (Potapov et al.)"}
COLOR = {"Our B4 Phase 2":"#76A5C7", "Pauls 2020":"#E8CF75", "Lang 2020":"#DD845D", "Meta/Tolan 2023":"#CF91A8", "GFCH 2019":"#8FC8D8"}
EDGES = {"Ifran": np.arange(0, 50, 5, dtype=float), "Maamoura": np.arange(0, 25, 5, dtype=float), "Agadir": np.arange(0, 25, 5, dtype=float)}
raw = pd.read_csv(SOURCE)
print(f"Loaded {len(raw):,} product-shot rows from {SOURCE}")

## Build strict-common support

A shot is retained only when every product displayed for that forest has a finite prediction. RH95 consistency is checked across products.

In [ ]:
strict_parts, support_rows = [], []
for forest, products in PRODUCTS.items():
    s = raw[raw.forest.eq(forest) & raw["product"].isin(products)].copy()
    s = s[np.isfinite(s.rh95) & np.isfinite(s.prediction)]
    if s.duplicated(["product", "shot_id"]).any():
        raise RuntimeError(f"{forest}: duplicate product-shot rows")
    sets = [set(s.loc[s["product"].eq(p), "shot_id"].astype(str)) for p in products]
    common = set.intersection(*sets)
    s = s[s.shot_id.astype(str).isin(common)].copy()
    counts = s.groupby("shot_id").size()
    assert counts.eq(len(products)).all()
    spread = s.groupby("shot_id").rh95.agg(lambda x: float(x.max()-x.min()))
    assert spread.max() < 1e-6
    edges = EDGES[forest].copy(); edges[-1] = np.nextafter(edges[-1], np.inf)
    labels = [f"{int(a)}–{int(b)} m" for a,b in zip(EDGES[forest][:-1], EDGES[forest][1:])]
    s["height_class"] = pd.cut(s.rh95, edges, labels=labels, right=False, include_lowest=True)
    strict_parts.append(s)
    support_rows.append({"forest":forest, "products":len(products), "strict_common_n":len(common), "min_rh95":s.rh95.min(), "max_rh95":s.rh95.max()})
strict = pd.concat(strict_parts, ignore_index=True)
support = pd.DataFrame(support_rows)
support.to_csv(TABLES / "00_strict_common_support.csv", index=False)
display(support)

## Metrics by height class

MAE and RMSE describe error magnitude; bias is prediction minus GEDI RH95. R², correlation, slope and spread ratio are retained as diagnostics but are not over-interpreted in small classes.

In [ ]:
def metrics(g):
    y=g.rh95.to_numpy(float); p=g.prediction.to_numpy(float); e=p-y; n=len(g)
    sst=np.sum((y-y.mean())**2); corr=np.corrcoef(y,p)[0,1] if n>2 and np.std(y)>0 and np.std(p)>0 else np.nan
    slope=np.polyfit(y,p,1)[0] if n>2 and np.std(y)>0 else np.nan
    return pd.Series({"n":n,"mae":np.mean(np.abs(e)),"rmse":np.sqrt(np.mean(e**2)),"bias":np.mean(e),"median_error":np.median(e),"r2":1-np.sum(e**2)/sst if sst>0 else np.nan,"corr":corr,"slope":slope,"std_ratio":np.std(p,ddof=1)/np.std(y,ddof=1) if n>2 and np.std(y,ddof=1)>0 else np.nan})

summary=(strict.dropna(subset=["height_class"]).groupby(["forest","product","height_class"],observed=True).apply(metrics,include_groups=False).reset_index())
summary["product_label"]=summary["product"].map(LABEL)
summary.to_csv(TABLES / "01_metrics_by_height_class_strict_common.csv", index=False)

counts=(strict[["forest","shot_id","rh95","height_class"]].drop_duplicates(["forest","shot_id"]).dropna(subset=["height_class"]).groupby(["forest","height_class"],observed=True).size().rename("n").reset_index())
counts.to_csv(TABLES / "02_gedi_counts_by_height_class.csv", index=False)
display(summary.round(3))

## Direct comparison: Our model versus Pa24, L23 and T24

Grouped bars place all four products side by side within every GEDI RH95 class. The number above each class is the shared GEDI sample size; it is identical for all four bars because strict-common support is enforced.

In [ ]:
fig, axes = plt.subplots(3,1,figsize=(12.4,11.5),dpi=180)
for i,forest in enumerate(["Ifran","Maamoura","Agadir"]):
    ax=axes[i]
    sf=summary[summary.forest.eq(forest)]; cf=counts[counts.forest.eq(forest)]
    classes=list(cf.height_class.astype(str)); x=np.arange(len(classes))
    width=0.19; offsets=(np.arange(len(PRODUCTS[forest]))-(len(PRODUCTS[forest])-1)/2)*width
    ymax=0
    for offset,product in zip(offsets,PRODUCTS[forest]):
        z=sf[sf["product"].eq(product)].set_index(sf[sf["product"].eq(product)].height_class.astype(str)).reindex(classes)
        vals=z.mae.to_numpy(float); ymax=max(ymax,np.nanmax(vals))
        bars=ax.bar(x+offset,vals,width=width,label=LABEL[product],color=COLOR[product],edgecolor='0.35',linewidth=.55)
    ax.set_xticks(x,classes); ax.set_ylabel("MAE (m)"); ax.grid(axis='y',alpha=.25); ax.set_axisbelow(True)
    ax.set_title(f"{forest}",loc='left',fontweight='bold')
    for xx,nn in zip(x,cf.n): ax.text(xx,ymax*1.045,f"n={int(nn):,}",ha='center',va='bottom',fontsize=8)
    ax.set_ylim(0,ymax*1.18)
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=4,frameon=False,bbox_to_anchor=(.5,.005))
fig.supxlabel("GEDI RH95 height class",y=.045); fig.tight_layout(rect=(0,.065,1,1),h_pad=1.8)
for ext in ['pdf','svg','png']: fig.savefig(FIGURES/f"01_height_class_MAE_and_support.{ext}",dpi=300,bbox_inches='tight',facecolor='white')
plt.show()

## Secondary diagnostics

RMSE and signed bias are useful for the supplement. Negative bias in tall classes indicates vertical compression; low class counts make estimates less stable but do not, by themselves, prove the cause.

In [ ]:
fig, axes=plt.subplots(3,2,figsize=(12.2,10.5),dpi=180)
for i,forest in enumerate(["Ifran","Maamoura","Agadir"]):
    sf=summary[summary.forest.eq(forest)]; classes=list(counts[counts.forest.eq(forest)].height_class.astype(str)); x=np.arange(len(classes))
    for j,(metric,title) in enumerate([("rmse","RMSE (m)"),("bias","Signed error: prediction − GEDI RH95 (m)")]):
        ax=axes[i,j]
        for p in PRODUCTS[forest]:
            z=sf[sf["product"].eq(p)].set_index(sf[sf["product"].eq(p)].height_class.astype(str)).reindex(classes)
            ax.plot(x,z[metric],marker='o',lw=1.7,ms=4.5,label=LABEL[p],color=COLOR[p])
        if metric=='bias': ax.axhline(0,color='k',ls='--',lw=.9)
        ax.set_xticks(x,classes,rotation=22,ha='right'); ax.set_ylabel(title); ax.grid(axis='y',alpha=.25); ax.set_title(f"{forest}",loc='left',fontweight='bold')
handles,labels=axes[0,0].get_legend_handles_labels(); fig.legend(handles,labels,loc='lower center',ncol=5,frameon=False,bbox_to_anchor=(.5,.005))
fig.tight_layout(rect=(0,.06,1,1),h_pad=2)
for ext in ['pdf','svg','png']: fig.savefig(FIGURES/f"02_height_class_RMSE_bias.{ext}",dpi=300,bbox_inches='tight',facecolor='white')
plt.show()

## Direct numerical tables by forest

Each table reports MAE for the four products on the same shots. The minimum value in each row is highlighted.

In [ ]:
for forest in ["Ifran","Maamoura","Agadir"]:
    table=(summary[summary.forest.eq(forest)].pivot(index="height_class",columns="product_label",values="mae"))
    order=[LABEL[p] for p in PRODUCTS[forest]]
    table=table.reindex(columns=order)
    n_by_class=counts[counts.forest.eq(forest)].set_index("height_class")["n"]
    table.insert(0,"GEDI n",n_by_class.reindex(table.index).astype(int))
    table.to_csv(TABLES/f"04_{forest.lower()}_MAE_products_by_height_class.csv")
    display(Markdown(f"### {forest}"))
    display(table.style.format({c:"{:.2f}" for c in order}).highlight_min(axis=1,subset=order,color="#d8efd3"))

## Extended comparison including P21 (Potapov et al.)

P21 is included for Ifran and Maamoura using the five-product strict-common intersection. It is not evaluable at Agadir because valid coverage is insufficient. This extended support is smaller than the four-product support and is therefore reported separately.

In [ ]:
extended_parts=[]; extended_support=[]
for forest,products in EXTENDED_PRODUCTS.items():
    s=raw[raw.forest.eq(forest)&raw["product"].isin(products)].copy()
    s=s[np.isfinite(s.rh95)&np.isfinite(s.prediction)]
    sets=[set(s.loc[s["product"].eq(p),"shot_id"].astype(str)) for p in products]
    common=set.intersection(*sets)
    s=s[s.shot_id.astype(str).isin(common)].copy()
    edges=EDGES[forest].copy(); edges[-1]=np.nextafter(edges[-1],np.inf)
    labels=[f"{int(a)}–{int(b)} m" for a,b in zip(EDGES[forest][:-1],EDGES[forest][1:])]
    s["height_class"]=pd.cut(s.rh95,edges,labels=labels,right=False,include_lowest=True)
    extended_parts.append(s); extended_support.append({"forest":forest,"products":len(products),"n":len(common),"P21_status":"included" if "GFCH 2019" in products else "not evaluable"})
extended=pd.concat(extended_parts,ignore_index=True)
extended_metrics=(extended.dropna(subset=["height_class"]).groupby(["forest","product","height_class"],observed=True).apply(metrics,include_groups=False).reset_index())
extended_metrics["product_label"]=extended_metrics["product"].map(LABEL)
pd.DataFrame(extended_support).to_csv(TABLES/"05_extended_support_including_P21.csv",index=False)
extended_metrics.to_csv(TABLES/"06_extended_metrics_including_P21.csv",index=False)
display(pd.DataFrame(extended_support))

In [ ]:
from matplotlib.patches import Patch
fig,axes=plt.subplots(3,1,figsize=(12.4,11.5),dpi=180)
for i,forest in enumerate(["Ifran","Maamoura","Agadir"]):
    ax=axes[i]; sf=extended_metrics[extended_metrics.forest.eq(forest)]
    products=EXTENDED_PRODUCTS[forest]; classes=list(sf.height_class.astype(str).drop_duplicates()); x=np.arange(len(classes))
    width=.16; offsets=(np.arange(5)-2)*width; ymax=0
    for j,p in enumerate(["Our B4 Phase 2","Pauls 2020","Lang 2020","Meta/Tolan 2023","GFCH 2019"]):
        if p not in products: continue
        z=sf[sf["product"].eq(p)].set_index(sf[sf["product"].eq(p)].height_class.astype(str)).reindex(classes)
        vals=z.mae.to_numpy(float); ymax=max(ymax,np.nanmax(vals)); ax.bar(x+offsets[j],vals,width,color=COLOR[p],edgecolor='.35',linewidth=.5)
    nclass=(extended[extended.forest.eq(forest)][["shot_id","height_class"]].drop_duplicates().groupby("height_class",observed=True).size().reindex(classes))
    for xx,nn in zip(x,nclass): ax.text(xx,ymax*1.045,f"n={int(nn):,}",ha='center',fontsize=8)
    ax.set_ylim(0,ymax*1.18); ax.set_xticks(x,classes); ax.set_ylabel("MAE (m)"); ax.grid(axis='y',alpha=.25); ax.set_axisbelow(True); ax.set_title(forest,loc='left',fontweight='bold')
    if forest=="Agadir": ax.text(.985,.9,"P21: not evaluable",transform=ax.transAxes,ha='right',va='top',fontsize=9,bbox=dict(facecolor='white',edgecolor='.7'))
legend=[Patch(facecolor=COLOR[p],edgecolor='.35',label=LABEL[p]) for p in ["Our B4 Phase 2","Pauls 2020","Lang 2020","Meta/Tolan 2023","GFCH 2019"]]
fig.legend(handles=legend,loc='lower center',ncol=5,frameon=False,bbox_to_anchor=(.5,.005)); fig.supxlabel("GEDI RH95 height class",y=.045); fig.tight_layout(rect=(0,.065,1,1),h_pad=1.8)
for ext in ['pdf','svg','png']: fig.savefig(FIGURES/f"03_extended_height_class_MAE_including_P21.{ext}",dpi=300,bbox_inches='tight',facecolor='white')
plt.show()

## Descriptive winners and article-ready interpretation

In [ ]:
winner_rows=[]
for (forest,hc),g in summary.groupby(["forest","height_class"],observed=True):
    best=g.loc[g.mae.idxmin()]
    ours=g[g["product"].eq("Our B4 Phase 2")].iloc[0]
    pauls=g[g["product"].eq("Pauls 2020")]
    winner_rows.append({"forest":forest,"height_class":str(hc),"n":int(ours.n),"lowest_mae_product":LABEL[best["product"]],"lowest_mae":best.mae,"our_mae":ours.mae,"pauls_mae":pauls.mae.iloc[0] if len(pauls) else np.nan,"our_minus_pauls_mae":ours.mae-(pauls.mae.iloc[0] if len(pauls) else np.nan)})
winners=pd.DataFrame(winner_rows); winners.to_csv(TABLES/"03_classwise_winners_descriptive.csv",index=False); display(winners.round(3))

print("ARTICLE-READY WORDING:\n")
print("Class-wise comparisons showed that the ranking of canopy-height products varied with GEDI RH95. Our locally trained model did not have the lowest error in every height class, even when it achieved the best aggregate metrics. Errors increased and negative bias became stronger in the tallest classes, where GEDI support was also substantially smaller. The limited number of tall-canopy observations may contribute to uncertainty in these estimates, while the systematic negative bias indicates additional vertical compression. These results therefore support product-level, height-dependent interpretation rather than a claim of universal superiority.")
print(f"\nSaved tables: {TABLES}\nSaved figures: {FIGURES}")